# Экспорт модели CIFAR100 в формат ONNX

В этом ноутбуке мы загрузим предобученную модель CIFAR100 и экспортируем её в формат ONNX для использования в веб-приложении.

## Шаг 1: Импорт необходимых библиотек

In [ ]:
import torch
import os

## Шаг 2: Создание устройства (CPU или GPU)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

## Шаг 3: Загрузка модели

Выберите модель в соответствии с вашим вариантом:
- Четный номер в списке группы - cifar100_mobile
- Нечетный номер в списке группы - cifar100_resnet

In [ ]:
# Раскомментируйте нужную модель в соответствии с вашим вариантом

# Для четного номера в списке группы
model = torch.hub.load("chenyaofo/pytorch-cifar-models",
                      "cifar100_mobilenetv2_x0_5",
                      pretrained=True)

# Для нечетного номера в списке группы
# model = torch.hub.load("chenyaofo/pytorch-cifar-models",
#                      "cifar100_resnet20",
#                      pretrained=True)

print("Модель успешно загружена")

## Шаг 4: Загрузка модели на устройство

In [ ]:
model.to(device)
model.eval()  # Установка модели в режим оценки (не обучения)

## Шаг 5: Экспорт модели в формат ONNX

In [ ]:
# Создаем входной тензор для экспорта
x = torch.randn(1, 3, 32, 32, requires_grad=True).to(device)

# Путь для сохранения модели
onnx_model_path = "media/models/cifar100.onnx"

# Создаем директорию, если она не существует
os.makedirs(os.path.dirname(onnx_model_path), exist_ok=True)

# Экспортируем модель в формат ONNX
torch.onnx.export(model,  # модель
                 x,  # входной тензор
                 onnx_model_path,  # путь для сохранения
                 export_params=True,  # сохраняет веса обученных параметров
                 opset_version=9,  # версия ONNX
                 do_constant_folding=True,  # укорачивание констант для оптимизации
                 input_names=['input'],  # имя входного слоя
                 output_names=['output'],  # имя выходного слоя
                 dynamic_axes={'input': {0: 'batch_size'},  # динамичные оси
                             'output': {0: 'batch_size'}})

print(f"Модель успешно экспортирована в {onnx_model_path}")

## Проверка экспортированной модели

In [ ]:
import onnx

# Загружаем модель ONNX для проверки
onnx_model = onnx.load(onnx_model_path)

# Проверяем модель
onnx.checker.check_model(onnx_model)
print("Модель ONNX проверена и корректна!")